In [ ]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), 'src'))

import torch
import gym
import numpy as np
import matplotlib.pyplot as plt
from src.config import load_config
from src.utils import set_seed
from src.agent import A2CAgent
from src.model import ActorCritic

In [ ]:
# Load configuration
config = load_config('configs/a2c.yaml')
set_seed(config['seed'])

# Create environment
env = gym.make(config['env_name'])
num_inputs = env.observation_space.shape[0]
num_actions = env.action_space.n

# Create agent
agent = A2CAgent(num_inputs, num_actions, config)

print(f"Environment: {config['env_name']}")
print(f"State dim: {num_inputs}, Actions: {num_actions}")

In [ ]:
# Training loop
episode_rewards = []
losses = []

state = env.reset()
episode_reward = 0
episode_count = 0

for step in range(config['total_timesteps']):
    # Collect rollout
    rollouts = {
        'states': [],
        'actions': [],
        'log_probs': [],
        'values': [],
        'rewards': [],
        'dones': []
    }
    
    for _ in range(config['num_steps']):
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        action, log_prob, value = agent.model.get_action(state_tensor)
        
        next_state, reward, done, _ = env.step(action.item())
        
        rollouts['states'].append(state_tensor)
        rollouts['actions'].append(action)
        rollouts['log_probs'].append(log_prob)
        rollouts['values'].append(value)
        rollouts['rewards'].append(reward)
        rollouts['dones'].append(done)
        
        episode_reward += reward
        state = next_state
        
        if done:
            episode_rewards.append(episode_reward)
            episode_reward = 0
            episode_count += 1
            state = env.reset()
    
    # Update agent
    loss_info = agent.update(rollouts)
    losses.append(loss_info)
    
    if step % 1000 == 0:
        print(f"Step {step}, Episodes: {episode_count}, Avg Reward: {np.mean(episode_rewards[-10:]):.2f}")

env.close()

In [ ]:
# Plot training results
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.plot(episode_rewards)
plt.title('Episode Rewards')
plt.xlabel('Episode')
plt.ylabel('Reward')

plt.subplot(1, 3, 2)
total_losses = [l['total_loss'] for l in losses]
plt.plot(total_losses)
plt.title('Total Loss')
plt.xlabel('Update Step')
plt.ylabel('Loss')

plt.subplot(1, 3, 3)
policy_losses = [l['policy_loss'] for l in losses]
value_losses = [l['value_loss'] for l in losses]
plt.plot(policy_losses, label='Policy Loss')
plt.plot(value_losses, label='Value Loss')
plt.title('Policy and Value Losses')
plt.xlabel('Update Step')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Evaluation
def evaluate_agent(agent, env_name, num_episodes=10):
    env = gym.make(env_name)
    rewards = []
    for _ in range(num_episodes):
        state = env.reset()
        episode_reward = 0
        done = False
        while not done:
            state_tensor = torch.FloatTensor(state).unsqueeze(0)
            with torch.no_grad():
                action_logits, _ = agent.model(state_tensor)
                action = torch.argmax(action_logits, dim=1).item()
            state, reward, done, _ = env.step(action)
            episode_reward += reward
        rewards.append(episode_reward)
    env.close()
    return np.mean(rewards), np.std(rewards)

mean_reward, std_reward = evaluate_agent(agent, config['env_name'])
print(f"Evaluation: Mean Reward = {mean_reward:.2f} ± {std_reward:.2f}")